# HyperParameter Tuning using Keras Tuner ->

## Hyperparameters :
##### 1. How many number of hidden layers we should have?
##### 2. How many number of neurons we should have in hidden layers?
##### 3. Learning Rate

### Keras Tuner- Decide Number of Hidden Layers And Neuron In Neural Network

In [7]:
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from keras_tuner.tuners import RandomSearch

### Load the data file 

In [8]:
df=pd.read_csv('../data/Real_Combine.csv')

df.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [9]:
X=df.iloc[:,:-1] ## independent features
y=df.iloc[:,-1] ## dependent features

## HyperParameter Tuning (on multiple hyperParameter values in the given range)

### Tuning / Selection of :
##### 1. No. of hidden layers
##### 2. No. of neurons in a layer 
##### 3. Learning rate


In [10]:
def build_model(hp):
    model = keras.Sequential() ##selecting / choosing the model type
    for i in range(hp.Int('num_layers', 2, 20)): ## ranges of layers defined
        model.add(layers.Dense(units=hp.Int('units_' + str(i),
                                            min_value=32, ## min. no. of neurons in a layer
                                            max_value=512, ## max. no. of neurons in a layer
                                            step=32),
                               activation='relu')) ## activation function (relu performs good in hidden layer so, proceding with that)
    model.add(layers.Dense(1, activation='linear')) ## output layer with single neurons (since it is a regression model thus , single o/p neuron)
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])), ## Learning Rates selection from (0.01, 0.001, 0.0001)
        loss='mean_absolute_error',
        metrics=['mean_absolute_error']) ## For regression model -> mean absolute error is calculated insted of accuracy (unlike classification)
    return model

In [11]:
tuner = RandomSearch(
    build_model,
    objective='val_mean_absolute_error',
    max_trials=5,
    executions_per_trial=3,
    directory='project',
    project_name='Air Quality Index')

2025-09-01 13:39:25.314893: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-09-01 13:39:25.315237: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-09-01 13:39:25.315277: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
2025-09-01 13:39:25.315376: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-01 13:39:25.315492: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [12]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 20, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


### Training and Testing data preperation

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [14]:
tuner.search(X_train, y_train,
             epochs=5,
             validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 09s]
val_mean_absolute_error: 2341.2060546875

Best val_mean_absolute_error So Far: 62.105857849121094
Total elapsed time: 00h 00m 48s


### mean_absolute_error (in regression model) should be as low as possible for a better model